In [23]:
# STEP 1: LOAD DATA

import pandas as pd

df = pd.read_csv('dataset.csv', engine='python', on_bad_lines='skip')

print("Columns:\n", df.columns)

Columns:
 Index(['id', 'asins', 'brand', 'categories', 'colors', 'dateAdded',
       'dateUpdated', 'dimension', 'ean', 'keys', 'manufacturer',
       'manufacturerNumber', 'name', 'prices', 'reviews.date',
       'reviews.doRecommend', 'reviews.numHelpful', 'reviews.rating',
       'reviews.sourceURLs', 'reviews.text', 'reviews.title',
       'reviews.userCity', 'reviews.userProvince', 'reviews.username', 'sizes',
       'upc', 'weight'],
      dtype='object')


In [24]:
# STEP 2: USE TEXT COLUMN ONLY

text_col = None
for col in df.columns:
    if 'text' in col.lower():
        text_col = col
        break

print("Using Text Column:", text_col)

df = df[[text_col]]
df = df.dropna()

print("Shape after cleaning:", df.shape)


Using Text Column: reviews.text
Shape after cleaning: (13, 1)


In [25]:
# STEP 3: CREATE TARGET (IMPORTANT FIX)


def create_sentiment(text):
    text = str(text).lower()

    if any(word in text for word in ['bad', 'worst', 'terrible', 'poor']):
        return 'negative'
    elif any(word in text for word in ['ok', 'average', 'fine']):
        return 'neutral'
    else:
        return 'positive'

df['sentiment'] = df[text_col].apply(create_sentiment)

In [26]:
# STEP 4: NLP PREPROCESSING

import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [stemmer.stem(w) for w in words]

    return " ".join(words)

df['clean_text'] = df[text_col].apply(clean_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [27]:

# STEP 5: TRAIN TEST SPLIT

from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['sentiment']

print("Final rows:", len(df))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Final rows: 13


In [28]:
# STEP 6: FEATURE ENGINEERING

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

bow = CountVectorizer(max_features=5000)
tfidf = TfidfVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [29]:
# STEP 7: MODELS

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, X_train, X_test, name):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred, average='weighted'))
    print("Recall:", recall_score(y_test, pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, pred, average='weighted'))

In [30]:
# STEP 8: RUN MODELS

evaluate(LogisticRegression(max_iter=200), X_train_bow, X_test_bow, "LR (BoW)")
evaluate(LogisticRegression(max_iter=200), X_train_tfidf, X_test_tfidf, "LR (TF-IDF)")

evaluate(MultinomialNB(), X_train_bow, X_test_bow, "NB (BoW)")
evaluate(MultinomialNB(), X_train_tfidf, X_test_tfidf, "NB (TF-IDF)")

evaluate(DecisionTreeClassifier(), X_train_bow, X_test_bow, "DT (BoW)")
evaluate(DecisionTreeClassifier(), X_train_tfidf, X_test_tfidf, "DT (TF-IDF)")


LR (BoW)
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0

LR (TF-IDF)
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0

NB (BoW)
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0

NB (TF-IDF)
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0

DT (BoW)
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0

DT (TF-IDF)
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
